# Feature Engineering — Credit Risk

Based on EDA findings, we clean and transform the data for model training.

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
print(f'Original shape: {df.shape}')
df.head()

Original shape: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## 1. Remove Invalid Records

EDA found: 1 client with age=0, 13 with age > 100.

In [19]:
print(f'Before: {len(df)}')
df = df[(df['age'] >= 18) & (df['age'] <= 100)]
print(f'After removing invalid ages: {len(df)}')
print(f'Removed: {150000 - len(df)} rows')

Before: 150000
After removing invalid ages: 149986
Removed: 14 rows


## 2. Handle Missing Values

- MonthlyIncome: 20% missing — fill with median + create flag
- NumberOfDependents: 2.6% missing — fill with median

In [20]:
# Create flag BEFORE filling (the fact that income is missing may be predictive)
df['income_missing'] = df['MonthlyIncome'].isnull().astype(int)
print(f'Clients with missing income: {df["income_missing"].sum()}')

# Fill with median
income_median = df['MonthlyIncome'].median()
dependents_median = df['NumberOfDependents'].median()

print(f'MonthlyIncome median: {income_median}')
print(f'NumberOfDependents median: {dependents_median}')

df['MonthlyIncome'].fillna(income_median, inplace=True)
df['NumberOfDependents'].fillna(dependents_median, inplace=True)

# Verify no more nulls
print(f'\nRemaining nulls: {df.isnull().sum().sum()}')

Clients with missing income: 29724
MonthlyIncome median: 5400.0
NumberOfDependents median: 0.0

Remaining nulls: 0


## 3. Cap Extreme Outliers

EDA found extreme values in RevolvingUtilization and DebtRatio.

In [21]:
# RevolvingUtilization: cap at 1.5 (above 1.0 means over credit limit, above 1.5 is likely error)
df["RevolvingUtilizationOfUnsecuredLines"]

print(f'RevolvingUtilization > 1.5 before cap: {(df["RevolvingUtilizationOfUnsecuredLines"] > 1.5).sum()}')
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.5)

# DebtRatio: cap at 99th percentile
debt_cap = df['DebtRatio'].quantile(0.99)
print(f'DebtRatio 99th percentile: {debt_cap:.2f}')
df['DebtRatio'] = df['DebtRatio'].clip(upper=debt_cap)

print('\nAfter capping:')
print(f'  RevolvingUtilization max: {df["RevolvingUtilizationOfUnsecuredLines"].max()}')
print(f'  DebtRatio max: {df["DebtRatio"].max():.2f}')

RevolvingUtilization > 1.5 before cap: 600
DebtRatio 99th percentile: 4979.60

After capping:
  RevolvingUtilization max: 1.5
  DebtRatio max: 4979.60


## 4. Combine Late Payment Columns

EDA showed correlation of 0.98-0.99 between the 3 late payment columns.
Combining them into one reduces redundancy.

In [22]:
df['total_late_payments'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

print('total_late_payments distribution:')
print(df['total_late_payments'].describe())
print(f'\nClients with 0 late payments: {(df["total_late_payments"] == 0).sum()}')
print(f'Clients with 1+ late payments: {(df["total_late_payments"] > 0).sum()}')

total_late_payments distribution:
count    149986.000000
mean          0.927460
std          12.466783
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         294.000000
Name: total_late_payments, dtype: float64

Clients with 0 late payments: 119625
Clients with 1+ late payments: 30361


## 5. Create Additional Features

In [23]:
# Has any late payment (binary: 0 or 1)
df['has_late_payment'] = (df['total_late_payments'] > 0).astype(int)

# Has severe late payment (90+ days)
df['has_severe_late'] = (df['NumberOfTimes90DaysLate'] > 0).astype(int)

print('New features:')
print(f'  has_late_payment: {df["has_late_payment"].mean():.2%} of clients')
print(f'  has_severe_late: {df["has_severe_late"].mean():.2%} of clients')

New features:
  has_late_payment: 20.24% of clients
  has_severe_late: 5.56% of clients


## 6. Select Final Features

Drop the original late payment columns (replaced by total_late_payments).

In [24]:
columns_to_drop = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
]

df = df.drop(columns=columns_to_drop)

print(f'Final shape: {df.shape}')
print(f'\nFinal columns:')
for col in df.columns:
    print(f'  {col}')

Final shape: (149986, 12)

Final columns:
  SeriousDlqin2yrs
  RevolvingUtilizationOfUnsecuredLines
  age
  DebtRatio
  MonthlyIncome
  NumberOfOpenCreditLinesAndLoans
  NumberRealEstateLoansOrLines
  NumberOfDependents
  income_missing
  total_late_payments
  has_late_payment
  has_severe_late


## 7. Verify Final Dataset

In [25]:
print('Nulls:', df.isnull().sum().sum())
print('Infinites:', np.isinf(df.select_dtypes(include=np.number)).sum().sum())
print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["SeriousDlqin2yrs"].value_counts()}')
print(f'\nDefault rate: {df["SeriousDlqin2yrs"].mean():.2%}')

df.describe().T

Nulls: 0
Infinites: 0
Shape: (149986, 12)
Target distribution:
SeriousDlqin2yrs
0    139961
1     10025
Name: count, dtype: int64

Default rate: 6.68%


,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,149986.0,0.066840,0.249745,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,149986.0,0.322941,0.358628,0.0,0.029878,0.154234,0.559053,1.5
age,149986.0,52.291101,14.764163,21.0,41.000000,52.000000,63.000000,99.0
DebtRatio,149986.0,316.548985,907.008053,0.0,0.175075,0.366503,0.868100,4979.6
MonthlyIncome,149986.0,6418.676216,12890.964643,0.0,3903.250000,5400.000000,7400.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,149986.0,8.452922,5.146013,0.0,5.000000,8.000000,11.000000,58.0
NumberRealEstateLoansOrLines,149986.0,1.018295,1.129787,0.0,0.000000,1.000000,2.000000,54.0
NumberOfDependents,149986.0,0.737462,1.107048,0.0,0.000000,0.000000,1.000000,20.0
income_missing,149986.0,0.198178,0.398629,0.0,0.000000,0.000000,0.000000,1.0
total_late_payments,149986.0,0.927460,12.466783,0.0,0.000000,0.000000,0.000000,294.0


## 8. Save Processed Dataset

In [26]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/credit_risk_clean.csv', index=False)
print(f'Saved to data/processed/credit_risk_clean.csv')
print(f'Shape: {df.shape}')

Saved to data/processed/credit_risk_clean.csv
Shape: (149986, 12)


## 9. Summary

Transformations applied:

1. Removed clients with age < 18 or age > 100
2. Created `income_missing` flag before filling MonthlyIncome nulls with median
3. Filled NumberOfDependents nulls with median
4. Capped RevolvingUtilization at 1.5
5. Capped DebtRatio at 99th percentile
6. Combined 3 late payment columns into `total_late_payments`
7. Created `has_late_payment` (binary)
8. Created `has_severe_late` (binary — 90+ days)
9. Dropped original late payment columns